# CUDA Streams

Design an experiment for asynchronous CUDA execution and overlap.

## Objectives

Distinguish stream ordering, host asynchrony, synchronization, and genuine overlap.

## Background

CUDA streams order operations within a stream while potentially allowing independent work in different streams to overlap.

## Prediction

CUDA operations issued from Python are normally asynchronous with respect to the host. A Python call can return after work has been placed into a CUDA stream, before the GPU has completed that work.

For a sequence of matrix multiplications in one CUDA stream, we should therefore distinguish three measurements:

1. **host enqueue time**: how long Python takes to submit the operations;
2. **GPU elapsed time**: how long the operations take on the GPU, measured with CUDA events;
3. **synchronized wall time**: how long the host observes from the first submission until all submitted work has completed.

Because all matrix multiplications are submitted to the same stream, they should execute in issue order. The host enqueue time should be substantially shorter than the synchronized wall time when the queued GPU workload is large enough.

Immediately after submission, the final CUDA event may still be incomplete. Waiting for that event should account for most of the difference between enqueue time and synchronized wall time.

CUDA-event time and host wall time need not be identical. They use different clocks and include different boundaries:

- CUDA events measure elapsed device work between two positions in a stream;
- synchronized wall time also includes Python submission overhead and the host's synchronization call.

This first experiment establishes timing and ordering semantics only. It does not demonstrate concurrent execution or overlap between streams.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
from pprint import pprint

import torch

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "CUDA stream experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
device_properties = torch.cuda.get_device_properties(device_index)

stream_environment = {
    "device_index": device_index,
    "device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        device_properties.major,
        device_properties.minor,
    ),
    "multiprocessor_count": device_properties.multi_processor_count,
    "default_stream": str(torch.cuda.default_stream(device)),
}

pprint(stream_environment)

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)
{'compute_capability': (12, 1),
 'default_stream': '<torch.cuda.Stream device=cuda:0 cuda_stream=0x0>',
 'device_index': 0,
 'device_name': 'NVIDIA GB10',
 'multiprocessor_count': 48}


### Host submission versus GPU completion

The default CUDA stream orders operations submitted to it. The following experiment submits a sequence of matrix multiplications to that stream.

Before each measured trial, the host synchronizes with the device. This prevents unfinished work from an earlier trial from contaminating the measurement.

Two CUDA events bracket the matrix multiplications:

- the start event is recorded before the first multiplication;
- the end event is recorded after the final multiplication.

Recording an event is itself asynchronous. Calling `query()` on the end event immediately after submission tells us whether the GPU has already reached that point without forcing synchronization.

The experiment then waits for the end event and reports:

- Python submission time;
- additional time spent waiting;
- total synchronized wall time;
- CUDA-event elapsed time.

The workload reuses preallocated output storage so allocation is not part of each matrix multiplication.

In [3]:
from time import perf_counter

import pandas as pd


MATRIX_SIZE = 4096
MATRIX_MULTIPLICATIONS = 32
DTYPE = torch.float32

generator = torch.Generator(device=device)
generator.manual_seed(20260802)

left = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
right = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
result = torch.empty_like(left)

matrix_bytes = left.numel() * left.element_size()
operation_flops = 2 * MATRIX_SIZE**3

workload_info = {
    "matrix_size": MATRIX_SIZE,
    "dtype": str(DTYPE),
    "bytes_per_matrix_mib": matrix_bytes / 1024**2,
    "matrix_multiplications": MATRIX_MULTIPLICATIONS,
    "flops_per_multiplication": operation_flops,
    "total_submitted_flops": operation_flops * MATRIX_MULTIPLICATIONS,
}

pprint(workload_info)

{'bytes_per_matrix_mib': 64.0,
 'dtype': 'torch.float32',
 'flops_per_multiplication': 137438953472,
 'matrix_multiplications': 32,
 'matrix_size': 4096,
 'total_submitted_flops': 4398046511104}


In [4]:
WARMUP_MULTIPLICATIONS = 4

for _ in range(WARMUP_MULTIPLICATIONS):
    torch.mm(left, right, out=result)

torch.cuda.synchronize(device)

print(
    f"Completed {WARMUP_MULTIPLICATIONS} warm-up matrix multiplications "
    "and synchronized the device."
)

Completed 4 warm-up matrix multiplications and synchronized the device.


In [ ]:
def measure_default_stream_trial(trial: int) -> dict[str, float | int | bool]:
    torch.cuda.synchronize(device)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    start_event.record()

    for _ in range(MATRIX_MULTIPLICATIONS):
        torch.mm(left, right, out=result)

    end_event.record()

    enqueue_end = perf_counter()
    complete_immediately_after_enqueue = end_event.query()

    end_event.synchronize()
    wall_end = perf_counter()

    enqueue_ms = (enqueue_end - wall_start) * 1_000
    synchronized_wall_ms = (wall_end - wall_start) * 1_000
    synchronization_wait_ms = (wall_end - enqueue_end) * 1_000
    gpu_elapsed_ms = start_event.elapsed_time(end_event)

    return {
        "trial": trial,
        "enqueue_ms": enqueue_ms,
        "synchronization_wait_ms": synchronization_wait_ms,
        "synchronized_wall_ms": synchronized_wall_ms,
        "gpu_elapsed_ms": gpu_elapsed_ms,
        "complete_after_enqueue": complete_immediately_after_enqueue,
        "enqueue_fraction": enqueue_ms / synchronized_wall_ms,
    }


TRIALS = 7

timing_results = pd.DataFrame(
    measure_default_stream_trial(trial) for trial in range(1, TRIALS + 1)
)

timing_results.round(
    {
        "enqueue_ms": 3,
        "synchronization_wait_ms": 3,
        "synchronized_wall_ms": 3,
        "gpu_elapsed_ms": 3,
        "enqueue_fraction": 4,
    }
)

,trial,enqueue_ms,synchronization_wait_ms,synchronized_wall_ms,gpu_elapsed_ms,complete_after_enqueue,enqueue_fraction
0,1,1.680,244.131,245.810,245.675,False,0.0068
1,2,0.160,252.753,252.913,252.899,False,0.0006
2,3,0.153,253.287,253.440,253.430,False,0.0006
3,4,0.151,254.116,254.268,254.258,False,0.0006
4,5,0.150,254.351,254.500,254.491,False,0.0006
5,6,0.152,252.129,252.281,252.271,False,0.0006
6,7,0.150,250.903,251.053,251.044,False,0.0006


In [ ]:
timing_summary = (
    timing_results[
        [
            "enqueue_ms",
            "synchronization_wait_ms",
            "synchronized_wall_ms",
            "gpu_elapsed_ms",
            "enqueue_fraction",
        ]
    ]
    .agg(["median", "min", "max"])
    .T
)

completion_counts = (
    timing_results["complete_after_enqueue"]
    .value_counts(dropna=False)
    .rename_axis("complete_after_enqueue")
    .to_frame("trials")
)

display(timing_summary.round(4))
display(completion_counts)

median_enqueue_ms = timing_results["enqueue_ms"].median()
median_wait_ms = timing_results["synchronization_wait_ms"].median()
median_wall_ms = timing_results["synchronized_wall_ms"].median()
median_gpu_ms = timing_results["gpu_elapsed_ms"].median()

print(f"Median host enqueue time: {median_enqueue_ms:.3f} ms")
print(f"Median synchronization wait: {median_wait_ms:.3f} ms")
print(f"Median synchronized wall time: {median_wall_ms:.3f} ms")
print(f"Median CUDA-event GPU time: {median_gpu_ms:.3f} ms")
print(f"Median wall/event difference: {median_wall_ms - median_gpu_ms:.3f} ms")

,median,min,max
enqueue_ms,0.1517,0.1496,1.6795
synchronization_wait_ms,252.7532,244.1309,254.3506
synchronized_wall_ms,252.9130,245.8104,254.5001
gpu_elapsed_ms,252.8991,245.6749,254.4906
enqueue_fraction,0.0006,0.0006,0.0068


,trials
complete_after_enqueue,
False,7


Median host enqueue time: 0.152 ms
Median synchronization wait: 252.753 ms
Median synchronized wall time: 252.913 ms
Median CUDA-event GPU time: 252.899 ms
Median wall/event difference: 0.014 ms


### Independent work in one stream versus two streams

CUDA streams provide independent ordering domains. Operations in separate streams may execute concurrently when:

1. no dependency forces an ordering relationship;
2. the GPU can schedule both workloads concurrently;
3. the workloads do not individually consume all relevant execution resources.

The following experiment keeps the total arithmetic work constant:

- the single-stream case submits all matrix multiplications to one stream;
- the two-stream case submits half to each of two independent streams.

Each stream receives distinct input and output tensors. This avoids data hazards between streams.

A coordinating start event gives both worker streams the same logical release point. Each worker stream records a completion event after its final multiplication. A coordinating stream waits for both completion events before recording the overall end event.

The primary measurement is therefore the complete GPU makespan from coordinated release until both streams have finished.

A large `4096 × 4096` FP32 matrix multiplication is expected to occupy a substantial fraction of the GB10's compute resources. Separate streams make concurrent scheduling possible, but they may not produce a speedup if one multiplication already saturates the relevant hardware.

Prediction:

- both variants should perform the same total number of matrix multiplications;
- two-stream submission may remain inexpensive for the host;
- two streams may show little or no makespan reduction for this large GEMM workload;
- a lack of speedup would not prove that streams cannot overlap smaller or complementary operations.

In [7]:
worker_stream_a = torch.cuda.Stream(device=device)
worker_stream_b = torch.cuda.Stream(device=device)
coordination_stream = torch.cuda.Stream(device=device)

left_a = left
right_a = right
result_a = result

left_b = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
right_b = torch.randn(
    (MATRIX_SIZE, MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=generator,
)
result_b = torch.empty_like(left_b)

if MATRIX_MULTIPLICATIONS % 2 != 0:
    raise ValueError(
        "MATRIX_MULTIPLICATIONS must be even for an equal two-stream split"
    )

MULTIPLICATIONS_PER_STREAM = MATRIX_MULTIPLICATIONS // 2

stream_workload_info = {
    "total_matrix_multiplications": MATRIX_MULTIPLICATIONS,
    "single_stream_multiplications": MATRIX_MULTIPLICATIONS,
    "two_stream_multiplications_per_stream": MULTIPLICATIONS_PER_STREAM,
    "two_stream_total_multiplications": 2 * MULTIPLICATIONS_PER_STREAM,
}

pprint(stream_workload_info)

{'single_stream_multiplications': 32,
 'total_matrix_multiplications': 32,
 'two_stream_multiplications_per_stream': 16,
 'two_stream_total_multiplications': 32}


In [8]:
with torch.cuda.stream(worker_stream_a):
    torch.mm(left_a, right_a, out=result_a)

with torch.cuda.stream(worker_stream_b):
    torch.mm(left_b, right_b, out=result_b)

torch.cuda.synchronize(device)

print("Warmed both worker streams and synchronized the device.")

Warmed both worker streams and synchronized the device.


In [11]:
def measure_single_stream_makespan(
    trial: int,
) -> dict[str, float | int | str]:
    torch.cuda.synchronize(device)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    with torch.cuda.stream(worker_stream_a):
        start_event.record()

        for _ in range(MATRIX_MULTIPLICATIONS):
            torch.mm(left_a, right_a, out=result_a)

        end_event.record()

    enqueue_end = perf_counter()

    end_event.synchronize()
    wall_end = perf_counter()

    return {
        "trial": trial,
        "configuration": "one stream",
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": start_event.elapsed_time(end_event),
    }


def measure_two_stream_makespan(
    trial: int,
) -> dict[str, float | int | str]:
    torch.cuda.synchronize(device)

    release_event = torch.cuda.Event(enable_timing=True)
    stream_a_done = torch.cuda.Event()
    stream_b_done = torch.cuda.Event()
    makespan_end = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    with torch.cuda.stream(coordination_stream):
        release_event.record()

    worker_stream_a.wait_event(release_event)
    worker_stream_b.wait_event(release_event)

    with torch.cuda.stream(worker_stream_a):
        for _ in range(MULTIPLICATIONS_PER_STREAM):
            torch.mm(left_a, right_a, out=result_a)

        stream_a_done.record()

    with torch.cuda.stream(worker_stream_b):
        for _ in range(MULTIPLICATIONS_PER_STREAM):
            torch.mm(left_b, right_b, out=result_b)

        stream_b_done.record()

    coordination_stream.wait_event(stream_a_done)
    coordination_stream.wait_event(stream_b_done)

    with torch.cuda.stream(coordination_stream):
        makespan_end.record()

    enqueue_end = perf_counter()

    makespan_end.synchronize()
    wall_end = perf_counter()

    return {
        "trial": trial,
        "configuration": "two streams",
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": release_event.elapsed_time(makespan_end),
    }

In [12]:
STREAM_TRIALS = 7

stream_measurements: list[dict[str, float | int | str]] = []

for trial in range(1, STREAM_TRIALS + 1):
    if trial % 2 == 1:
        stream_measurements.append(measure_single_stream_makespan(trial))
        stream_measurements.append(measure_two_stream_makespan(trial))
    else:
        stream_measurements.append(measure_two_stream_makespan(trial))
        stream_measurements.append(measure_single_stream_makespan(trial))

stream_results = pd.DataFrame(stream_measurements)

stream_results.round(
    {
        "enqueue_ms": 3,
        "synchronized_wall_ms": 3,
        "gpu_makespan_ms": 3,
    }
)

,trial,configuration,enqueue_ms,synchronized_wall_ms,gpu_makespan_ms
0,1,one stream,1.492,247.108,246.931
1,1,two streams,0.222,248.967,248.940
2,2,two streams,0.185,248.428,248.413
3,2,one stream,0.155,253.628,253.614
4,3,one stream,0.156,253.096,253.082
5,3,two streams,0.181,248.639,248.624
6,4,two streams,0.179,249.638,249.624
7,4,one stream,0.156,253.938,253.924
8,5,one stream,0.158,255.282,255.268
9,5,two streams,0.180,249.657,249.643


In [ ]:
stream_summary = stream_results.groupby("configuration")[
    [
        "enqueue_ms",
        "synchronized_wall_ms",
        "gpu_makespan_ms",
    ]
].agg(["median", "min", "max"])

display(stream_summary.round(3))

median_makespans = stream_results.groupby("configuration")["gpu_makespan_ms"].median()

one_stream_ms = median_makespans["one stream"]
two_stream_ms = median_makespans["two streams"]

two_stream_speedup = one_stream_ms / two_stream_ms
makespan_change_percent = (two_stream_ms - one_stream_ms) / one_stream_ms * 100

comparison = pd.DataFrame(
    [
        {
            "one_stream_median_ms": one_stream_ms,
            "two_stream_median_ms": two_stream_ms,
            "two_stream_speedup": two_stream_speedup,
            "two_stream_makespan_change_percent": makespan_change_percent,
        }
    ]
)

display(comparison.round(4))

enqueue_ms               synchronized_wall_ms                    \
                  median    min    max               median      min      max   
configuration                                                                   
one stream         0.156  0.154  1.492              253.924  247.108  255.282   
two streams        0.181  0.179  0.222              249.638  248.428  249.874   

              gpu_makespan_ms                    
                       median      min      max  
configuration                                    
one stream            253.910  246.931  255.268  
two streams           249.624  248.413  249.861

,one_stream_median_ms,two_stream_median_ms,two_stream_speedup,two_stream_makespan_change_percent
0,253.9099,249.6245,1.0172,-1.6878


### Does workload size determine useful overlap?

The previous experiment used large matrix multiplications and found only a small two-stream improvement. This is consistent with each GEMM already occupying most of the GPU's useful compute capacity.

The next experiment varies matrix dimension while keeping the total nominal arithmetic work approximately constant.

For each matrix size, it compares:

- a single stream executing all multiplications;
- two streams executing half of the multiplications each.

Smaller matrix multiplications may use fewer thread blocks or otherwise expose less parallel work per kernel. In that case, two independent kernels may be able to occupy the GPU concurrently and reduce total makespan.

Prediction:

- large GEMMs should remain close to a `1×` two-stream speedup;
- intermediate or small GEMMs may exhibit a larger speedup;
- very small GEMMs may again show limited benefit because launch and scheduling overhead become significant.

A speedup demonstrates reduced aggregate makespan. It does not by itself identify the exact degree or mechanism of kernel overlap.

In [ ]:
TARGET_TOTAL_FLOPS = operation_flops * MATRIX_MULTIPLICATIONS

MATRIX_SIZES = [
    512,
    1024,
    2048,
    4096,
]

SIZE_SWEEP_TRIALS = 5


def even_multiplication_count_for_size(matrix_size: int) -> int:
    flops_per_multiplication = 2 * matrix_size**3
    approximate_count = max(
        2,
        round(TARGET_TOTAL_FLOPS / flops_per_multiplication),
    )

    if approximate_count % 2 != 0:
        approximate_count += 1

    return approximate_count


size_sweep_configuration = pd.DataFrame(
    {
        "matrix_size": MATRIX_SIZES,
        "multiplications": [
            even_multiplication_count_for_size(matrix_size)
            for matrix_size in MATRIX_SIZES
        ],
    }
)

size_sweep_configuration["flops_per_multiplication"] = (
    2 * size_sweep_configuration["matrix_size"] ** 3
)
size_sweep_configuration["total_nominal_flops"] = (
    size_sweep_configuration["multiplications"]
    * size_sweep_configuration["flops_per_multiplication"]
)
size_sweep_configuration["target_fraction"] = (
    size_sweep_configuration["total_nominal_flops"] / TARGET_TOTAL_FLOPS
)

size_sweep_configuration

,matrix_size,multiplications,flops_per_multiplication,total_nominal_flops,target_fraction
0,512,16384,268435456,4398046511104,1.0
1,1024,2048,2147483648,4398046511104,1.0
2,2048,256,17179869184,4398046511104,1.0
3,4096,32,137438953472,4398046511104,1.0


In [ ]:
def measure_stream_count_for_size(
    *,
    matrix_size: int,
    multiplication_count: int,
    stream_count: int,
    trial: int,
) -> dict[str, float | int]:
    if stream_count not in {1, 2}:
        raise ValueError("stream_count must be 1 or 2")

    if multiplication_count % stream_count != 0:
        raise ValueError("multiplication_count must divide evenly across streams")

    local_generator = torch.Generator(device=device)
    local_generator.manual_seed(20260803 + matrix_size)

    local_left_a = torch.randn(
        (matrix_size, matrix_size),
        device=device,
        dtype=DTYPE,
        generator=local_generator,
    )
    local_right_a = torch.randn(
        (matrix_size, matrix_size),
        device=device,
        dtype=DTYPE,
        generator=local_generator,
    )
    local_result_a = torch.empty_like(local_left_a)

    local_left_b = torch.randn(
        (matrix_size, matrix_size),
        device=device,
        dtype=DTYPE,
        generator=local_generator,
    )
    local_right_b = torch.randn(
        (matrix_size, matrix_size),
        device=device,
        dtype=DTYPE,
        generator=local_generator,
    )
    local_result_b = torch.empty_like(local_left_b)

    local_stream_a = torch.cuda.Stream(device=device)
    local_stream_b = torch.cuda.Stream(device=device)
    local_coordination_stream = torch.cuda.Stream(device=device)

    with torch.cuda.stream(local_stream_a):
        torch.mm(
            local_left_a,
            local_right_a,
            out=local_result_a,
        )

    with torch.cuda.stream(local_stream_b):
        torch.mm(
            local_left_b,
            local_right_b,
            out=local_result_b,
        )

    torch.cuda.synchronize(device)

    release_event = torch.cuda.Event(enable_timing=True)
    stream_a_done = torch.cuda.Event()
    stream_b_done = torch.cuda.Event()
    makespan_end = torch.cuda.Event(enable_timing=True)

    multiplications_per_stream = multiplication_count // stream_count

    wall_start = perf_counter()

    with torch.cuda.stream(local_coordination_stream):
        release_event.record()

    local_stream_a.wait_event(release_event)

    with torch.cuda.stream(local_stream_a):
        for _ in range(multiplications_per_stream):
            torch.mm(
                local_left_a,
                local_right_a,
                out=local_result_a,
            )

        stream_a_done.record()

    local_coordination_stream.wait_event(stream_a_done)

    if stream_count == 2:
        local_stream_b.wait_event(release_event)

        with torch.cuda.stream(local_stream_b):
            for _ in range(multiplications_per_stream):
                torch.mm(
                    local_left_b,
                    local_right_b,
                    out=local_result_b,
                )

            stream_b_done.record()

        local_coordination_stream.wait_event(stream_b_done)

    with torch.cuda.stream(local_coordination_stream):
        makespan_end.record()

    enqueue_end = perf_counter()

    makespan_end.synchronize()
    wall_end = perf_counter()

    return {
        "trial": trial,
        "matrix_size": matrix_size,
        "multiplication_count": multiplication_count,
        "stream_count": stream_count,
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": release_event.elapsed_time(makespan_end),
    }

In [16]:
size_sweep_measurements: list[dict[str, float | int]] = []

for configuration in size_sweep_configuration.itertuples(index=False):
    matrix_size = int(configuration.matrix_size)
    multiplication_count = int(configuration.multiplications)

    for trial in range(1, SIZE_SWEEP_TRIALS + 1):
        stream_order = (1, 2) if trial % 2 == 1 else (2, 1)

        for stream_count in stream_order:
            size_sweep_measurements.append(
                measure_stream_count_for_size(
                    matrix_size=matrix_size,
                    multiplication_count=multiplication_count,
                    stream_count=stream_count,
                    trial=trial,
                )
            )

size_sweep_results = pd.DataFrame(size_sweep_measurements)

size_sweep_results.round(
    {
        "enqueue_ms": 3,
        "synchronized_wall_ms": 3,
        "gpu_makespan_ms": 3,
    }
)

,trial,matrix_size,multiplication_count,stream_count,enqueue_ms,synchronized_wall_ms,gpu_makespan_ms
0,1,512,16384,1,491.135,506.722,506.672
1,1,512,16384,2,483.068,498.720,498.675
2,2,512,16384,2,482.081,497.724,497.689
3,2,512,16384,1,489.894,505.536,505.500
4,3,512,16384,1,488.016,503.671,503.632
5,3,512,16384,2,483.591,499.240,499.207
6,4,512,16384,2,484.376,500.012,499.972
7,4,512,16384,1,489.662,505.299,505.265
8,5,512,16384,1,488.889,504.491,504.456
9,5,512,16384,2,483.618,499.280,499.242


In [ ]:
size_sweep_medians = (
    size_sweep_results.groupby(["matrix_size", "multiplication_count", "stream_count"])[
        [
            "enqueue_ms",
            "synchronized_wall_ms",
            "gpu_makespan_ms",
        ]
    ]
    .median()
    .reset_index()
)

makespan_comparison = (
    size_sweep_medians.pivot(
        index=["matrix_size", "multiplication_count"],
        columns="stream_count",
        values="gpu_makespan_ms",
    )
    .rename(
        columns={
            1: "one_stream_ms",
            2: "two_stream_ms",
        }
    )
    .reset_index()
)

makespan_comparison["two_stream_speedup"] = (
    makespan_comparison["one_stream_ms"] / makespan_comparison["two_stream_ms"]
)

makespan_comparison["two_stream_change_percent"] = (
    (makespan_comparison["two_stream_ms"] - makespan_comparison["one_stream_ms"])
    / makespan_comparison["one_stream_ms"]
    * 100
)

display(size_sweep_medians.round(3))
display(makespan_comparison.round(4))

,matrix_size,multiplication_count,stream_count,enqueue_ms,synchronized_wall_ms,gpu_makespan_ms
0,512,16384,1,489.662,505.299,505.265
1,512,16384,2,483.591,499.240,499.207
2,1024,2048,1,220.865,294.394,294.349
3,1024,2048,2,217.246,290.945,290.899
4,2048,256,1,1.561,268.381,268.367
5,2048,256,2,1.583,252.056,252.042
6,4096,32,1,0.176,255.755,255.741
7,4096,32,2,0.186,251.492,251.478


stream_count,matrix_size,multiplication_count,one_stream_ms,two_stream_ms,two_stream_speedup,two_stream_change_percent
0,512,16384,505.2650,499.2071,1.0121,-1.1990
1,1024,2048,294.3489,290.8992,1.0119,-1.1720
2,2048,256,268.3667,252.0422,1.0648,-6.0829
3,4096,32,255.7408,251.4780,1.0170,-1.6669


### Stream independence versus an explicit dependency

Separate CUDA streams provide independent ordering domains, but explicit dependencies can impose ordering between them.

The previous size sweep found the largest two-stream makespan reduction for `2048 × 2048` matrix multiplications. This experiment uses that workload to compare:

1. **independent streams**: both workers may begin after a shared release event;
2. **dependent streams**: stream B must wait for stream A to finish.

Both cases perform the same number of matrix multiplications on the same tensors. Only the event dependency changes.

Each worker stream records its own start and completion events. This lets us inspect:

- overall GPU makespan;
- the duration observed by each worker stream;
- when each worker starts and finishes relative to the shared release event.

Prediction:

- independent streams should have overlapping worker intervals and reproduce the earlier makespan benefit;
- the explicit `stream_b.wait_event(stream_a_done)` dependency should move stream B's start until after stream A completes;
- the dependent configuration should have a makespan close to the single-stream result;
- separate stream objects alone do not imply concurrent execution when an event dependency serializes their work.

In [ ]:
DEPENDENCY_MATRIX_SIZE = 2048
DEPENDENCY_TOTAL_MULTIPLICATIONS = 256
DEPENDENCY_MULTIPLICATIONS_PER_STREAM = DEPENDENCY_TOTAL_MULTIPLICATIONS // 2
DEPENDENCY_TRIALS = 7

dependency_generator = torch.Generator(device=device)
dependency_generator.manual_seed(20260803)

dependency_left_a = torch.randn(
    (DEPENDENCY_MATRIX_SIZE, DEPENDENCY_MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=dependency_generator,
)
dependency_right_a = torch.randn(
    (DEPENDENCY_MATRIX_SIZE, DEPENDENCY_MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=dependency_generator,
)
dependency_result_a = torch.empty_like(dependency_left_a)

dependency_left_b = torch.randn(
    (DEPENDENCY_MATRIX_SIZE, DEPENDENCY_MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=dependency_generator,
)
dependency_right_b = torch.randn(
    (DEPENDENCY_MATRIX_SIZE, DEPENDENCY_MATRIX_SIZE),
    device=device,
    dtype=DTYPE,
    generator=dependency_generator,
)
dependency_result_b = torch.empty_like(dependency_left_b)

dependency_stream_a = torch.cuda.Stream(device=device)
dependency_stream_b = torch.cuda.Stream(device=device)
dependency_coordination_stream = torch.cuda.Stream(device=device)

with torch.cuda.stream(dependency_stream_a):
    torch.mm(
        dependency_left_a,
        dependency_right_a,
        out=dependency_result_a,
    )

with torch.cuda.stream(dependency_stream_b):
    torch.mm(
        dependency_left_b,
        dependency_right_b,
        out=dependency_result_b,
    )

torch.cuda.synchronize(device)

print(
    f"Configured {DEPENDENCY_TOTAL_MULTIPLICATIONS} total "
    f"{DEPENDENCY_MATRIX_SIZE} x {DEPENDENCY_MATRIX_SIZE} GEMMs."
)

Configured 256 total 2048 x 2048 GEMMs.


In [ ]:
def measure_dependency_trial(
    *,
    trial: int,
    dependent: bool,
) -> dict[str, float | int | str]:
    torch.cuda.synchronize(device)

    release = torch.cuda.Event(enable_timing=True)

    stream_a_start = torch.cuda.Event(enable_timing=True)
    stream_a_done = torch.cuda.Event(enable_timing=True)

    stream_b_start = torch.cuda.Event(enable_timing=True)
    stream_b_done = torch.cuda.Event(enable_timing=True)

    makespan_end = torch.cuda.Event(enable_timing=True)

    wall_start = perf_counter()

    with torch.cuda.stream(dependency_coordination_stream):
        release.record()

    dependency_stream_a.wait_event(release)

    with torch.cuda.stream(dependency_stream_a):
        stream_a_start.record()

        for _ in range(DEPENDENCY_MULTIPLICATIONS_PER_STREAM):
            torch.mm(
                dependency_left_a,
                dependency_right_a,
                out=dependency_result_a,
            )

        stream_a_done.record()

    dependency_stream_b.wait_event(release)

    if dependent:
        dependency_stream_b.wait_event(stream_a_done)

    with torch.cuda.stream(dependency_stream_b):
        stream_b_start.record()

        for _ in range(DEPENDENCY_MULTIPLICATIONS_PER_STREAM):
            torch.mm(
                dependency_left_b,
                dependency_right_b,
                out=dependency_result_b,
            )

        stream_b_done.record()

    dependency_coordination_stream.wait_event(stream_a_done)
    dependency_coordination_stream.wait_event(stream_b_done)

    with torch.cuda.stream(dependency_coordination_stream):
        makespan_end.record()

    enqueue_end = perf_counter()

    makespan_end.synchronize()
    wall_end = perf_counter()

    stream_a_start_ms = release.elapsed_time(stream_a_start)
    stream_a_end_ms = release.elapsed_time(stream_a_done)
    stream_b_start_ms = release.elapsed_time(stream_b_start)
    stream_b_end_ms = release.elapsed_time(stream_b_done)

    interval_overlap_ms = max(
        0.0,
        min(stream_a_end_ms, stream_b_end_ms)
        - max(stream_a_start_ms, stream_b_start_ms),
    )

    return {
        "trial": trial,
        "configuration": ("dependent streams" if dependent else "independent streams"),
        "enqueue_ms": (enqueue_end - wall_start) * 1_000,
        "synchronized_wall_ms": (wall_end - wall_start) * 1_000,
        "gpu_makespan_ms": release.elapsed_time(makespan_end),
        "stream_a_start_ms": stream_a_start_ms,
        "stream_a_end_ms": stream_a_end_ms,
        "stream_a_duration_ms": stream_a_start.elapsed_time(stream_a_done),
        "stream_b_start_ms": stream_b_start_ms,
        "stream_b_end_ms": stream_b_end_ms,
        "stream_b_duration_ms": stream_b_start.elapsed_time(stream_b_done),
        "interval_overlap_ms": interval_overlap_ms,
    }

In [ ]:
dependency_measurements: list[dict[str, float | int | str]] = []

for trial in range(1, DEPENDENCY_TRIALS + 1):
    dependency_order = (False, True) if trial % 2 == 1 else (True, False)

    for dependent in dependency_order:
        dependency_measurements.append(
            measure_dependency_trial(
                trial=trial,
                dependent=dependent,
            )
        )

dependency_results = pd.DataFrame(dependency_measurements)

dependency_results.round(3)

,trial,configuration,enqueue_ms,synchronized_wall_ms,gpu_makespan_ms,stream_a_start_ms,stream_a_end_ms,stream_a_duration_ms,stream_b_start_ms,stream_b_end_ms,stream_b_duration_ms,interval_overlap_ms
0,1,independent streams,3.023,254.366,254.223,1.176,253.250,252.073,2.088,254.220,252.132,251.162
1,1,dependent streams,1.638,269.006,268.988,0.012,134.535,134.523,134.538,268.985,134.447,0.000
2,2,dependent streams,1.668,268.252,268.237,0.011,133.911,133.900,133.913,268.233,134.320,0.000
3,2,independent streams,1.605,253.571,253.557,0.011,252.589,252.578,0.803,253.553,252.750,251.786
4,3,independent streams,1.597,252.480,252.466,0.010,251.481,251.471,0.799,252.463,251.663,250.682
5,3,dependent streams,1.587,269.104,269.090,0.010,134.601,134.590,134.603,269.087,134.483,0.000
6,4,dependent streams,1.585,268.749,268.736,0.010,134.049,134.038,134.051,268.732,134.681,0.000
7,4,independent streams,1.608,252.859,252.846,0.010,251.887,251.877,0.802,252.843,252.041,251.085
8,5,independent streams,1.595,252.117,252.103,0.010,251.143,251.132,0.804,252.099,251.295,250.338
9,5,dependent streams,1.595,268.039,268.026,0.010,134.081,134.071,134.084,268.022,133.939,0.000


In [ ]:
dependency_summary = dependency_results.groupby("configuration")[
    [
        "enqueue_ms",
        "synchronized_wall_ms",
        "gpu_makespan_ms",
        "stream_a_start_ms",
        "stream_a_end_ms",
        "stream_a_duration_ms",
        "stream_b_start_ms",
        "stream_b_end_ms",
        "stream_b_duration_ms",
        "interval_overlap_ms",
    ]
].agg(["median", "min", "max"])

display(dependency_summary.round(3))

dependency_medians = dependency_results.groupby("configuration")[
    [
        "gpu_makespan_ms",
        "stream_a_start_ms",
        "stream_a_end_ms",
        "stream_b_start_ms",
        "stream_b_end_ms",
        "interval_overlap_ms",
    ]
].median()

display(dependency_medians.round(3))

independent_makespan_ms = dependency_medians.loc[
    "independent streams",
    "gpu_makespan_ms",
]
dependent_makespan_ms = dependency_medians.loc[
    "dependent streams",
    "gpu_makespan_ms",
]

dependency_cost = pd.DataFrame(
    [
        {
            "independent_makespan_ms": independent_makespan_ms,
            "dependent_makespan_ms": dependent_makespan_ms,
            "dependency_slowdown": (dependent_makespan_ms / independent_makespan_ms),
            "dependency_added_makespan_percent": (
                (dependent_makespan_ms - independent_makespan_ms)
                / independent_makespan_ms
                * 100
            ),
        }
    ]
)

display(dependency_cost.round(4))

enqueue_ms               synchronized_wall_ms           \
                        median    min    max               median      min   
configuration                                                                
dependent streams        1.587  1.562  1.668              268.749  268.039   
independent streams      1.598  1.595  3.023              252.859  250.110   

                             gpu_makespan_ms                    \
                         max          median      min      max   
configuration                                                    
dependent streams    269.633         268.736  268.026  269.620   
independent streams  254.366         252.846  250.096  254.223   

                    stream_a_start_ms  ... stream_b_start_ms stream_b_end_ms  \
                               median  ...               max          median   
configuration                          ...                                     
dependent streams                0.01  ...           134.649         268.732   
independent streams              0.01  ...             2.088         252.843   

                                      stream_b_duration_ms                    \
                         min      max               median      min      max   
configuration                                                                  
dependent streams    268.022  269.617              134.447  133.939  134.968   
independent streams  250.093  254.220              252.041  249.300  252.750   

                    interval_overlap_ms                    
                                 median      min      max  
configuration                                              
dependent streams                 0.000    0.000    0.000  
independent streams             251.085  248.339  251.786  

[2 rows x 30 columns]

,gpu_makespan_ms,stream_a_start_ms,stream_a_end_ms,stream_b_start_ms,stream_b_end_ms,interval_overlap_ms
configuration,,,,,,
dependent streams,268.736,0.01,134.186,134.189,268.732,0.000
independent streams,252.846,0.01,251.887,0.803,252.843,251.085


,independent_makespan_ms,dependent_makespan_ms,dependency_slowdown,dependency_added_makespan_percent
0,252.8464,268.7357,1.0628,6.2842


## Observations

### Host submission versus completion

For 32 `4096 × 4096` FP32 matrix multiplications in the default stream:

- median host enqueue time was `0.152 ms`;
- median synchronization wait was `252.753 ms`;
- median synchronized wall time was `252.913 ms`;
- median CUDA-event elapsed time was `252.899 ms`.

The final event was not complete immediately after submission in any measured trial. Python therefore returned while nearly all GPU execution remained outstanding.

CUDA-event time and synchronized wall time differed by only `0.014 ms` at the median for this long-running workload.

### One stream versus two streams

Splitting the same 32 large GEMMs equally across two independent streams changed median GPU makespan from:

- `253.910 ms` with one stream;
- to `249.625 ms` with two streams.

This was a `1.017×` speedup, or a `1.69%` makespan reduction. Separate streams therefore provided little benefit for the large-GEMM workload.

### Matrix-size sweep

Keeping nominal arithmetic work approximately constant produced:

| Matrix size | Multiplications | One stream | Two streams | Speedup |
|---:|---:|---:|---:|---:|
| 512 | 16,384 | 505.265 ms | 499.207 ms | 1.012× |
| 1024 | 2,048 | 294.349 ms | 290.899 ms | 1.012× |
| 2048 | 256 | 268.367 ms | 252.042 ms | 1.065× |
| 4096 | 32 | 255.741 ms | 251.478 ms | 1.017× |

The strongest observed two-stream benefit occurred at matrix size 2048, where makespan fell by `6.08%`.

Equal nominal FLOP counts did not produce equal execution times. The smaller-GEMM workloads required many more launches and completed less efficiently.

### Explicit cross-stream dependency

For 256 `2048 × 2048` GEMMs split equally across two streams:

- independent-stream median makespan was `252.846 ms`;
- dependency-serialized median makespan was `268.736 ms`;
- the explicit dependency added `6.28%` to makespan.

With independent streams, the recorded worker intervals overlapped for a median of `251.085 ms`.

With stream B waiting for stream A's completion event:

- stream A ended at `134.186 ms`;
- stream B started at `134.189 ms`;
- calculated interval overlap was `0.000 ms`.

The event dependency therefore serialized the stream intervals without requiring host synchronization.

## Explanation

A CUDA stream is an ordered queue. Operations submitted to one stream execute according to that stream's ordering rules, while submission from the host normally remains asynchronous.

The first experiment separated host submission from device completion. Python enqueued approximately 253 ms of GPU work in about 0.15 ms. The missing time appeared when the host synchronized with the completion event.

Multiple streams create separate ordering domains, but they do not create additional GPU hardware. Concurrent execution depends on whether independently schedulable workloads can share the device's finite resources.

The large `4096 × 4096` GEMMs gained little from two streams. This is consistent with each GEMM already using most of the relevant compute capacity, although the timing experiment did not directly measure occupancy.

The matrix-size sweep showed that useful overlap was workload-dependent rather than monotonic. The `2048 × 2048` case produced the largest makespan reduction. Smaller GEMMs were less efficient because equal nominal arithmetic required many more kernel launches and incurred more fixed per-operation cost.

The dependency experiment distinguished stream independence from stream concurrency. In the independent case, both streams had simultaneously active event intervals. In the dependent case, `stream_b.wait_event(stream_a_done)` prevented stream B from starting until stream A had completed.

The independent worker intervals overlapped for nearly the entire makespan, but each half-workload ran much longer under contention than it did alone. This explains why overlap improved throughput by only about 6% rather than approaching a twofold speedup.

CUDA event intervals establish stream-level timing and ordering. They do not reveal the precise kernel execution timeline, SM occupancy, or which kernel phases executed concurrently. Those questions require profiling with tools such as Nsight Systems or Nsight Compute.

## Connection to LLMs

LLM inference systems use CUDA streams to coordinate work with different dependency structures, including:

- model execution for separate requests;
- asynchronous memory copies;
- collective communication;
- attention and expert-routing work;
- preprocessing and postprocessing kernels;
- pipeline stages across devices.

Streams are most useful when workloads are independent and leave complementary or unused device resources. Running two already-saturating matrix multiplications in separate streams does not double throughput.

Events provide device-side coordination without forcing the CPU to wait. An inference runtime can therefore express dependencies such as:

1. a transfer must complete before a kernel consumes its data;
2. communication must complete before a later model layer begins;
3. independent requests may proceed in separate streams;
4. a coordinating stream may wait for several workers before continuing.

The measured results also illustrate the difference between latency and throughput. Concurrent streams reduced aggregate makespan, but contention made each individual worker interval longer. An inference server may accept higher per-request latency when the resulting overlap improves total request throughput.

## Further Exploration

Potential extensions include:

- capture the experiments with Nsight Systems to inspect the actual kernel timeline;
- compare FP32, TF32, FP16, and BF16 GEMMs;
- test more than two streams;
- measure compute overlap with asynchronous memory operations;
- compare event dependencies with host-side synchronization;
- investigate stream priorities;
- repeat the size sweep with CUDA Graph replay to reduce Python launch overhead;
- evaluate representative LLM operations rather than isolated square GEMMs.

These extensions should preserve the distinction between:

- overlapping stream intervals;
- concurrent kernel execution;
- reduced aggregate makespan;
- improved application-level throughput.